In [ ]:
!pip install pymongo dnspython pandas numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 17.5 MB/s eta 0:00:00


In [ ]:
from __future__ import annotations
import argparse
import json
import logging
import math
from pathlib import Path
from datetime import datetime
import pandas as pd
from pymongo import MongoClient
import pymongo

In [ ]:
LOG_PATH          = Path("logs/04_validate.log")
REPORT_PATH       = Path("logs/validation_report.txt")
INVALID_DOCS_PATH = Path("logs/invalid_docs.json")

MONGO_URI   = "mongodb+srv://<username>:<password>@cluster0.ikxi8.mongodb.net/?appName=Cluster0"
DB_NAME     = "wildfire_project"
COLLECTION  = "wildfires"

In [ ]:
REQUIRED_FIELDS = [
    "_id",
    "fire_name",
    "discovery_date",
    "final_size_acres",
    "size_log",
    "size_class",
    "ignition_cause",
    "location",
    "state",
]

In [ ]:
VALID_SIZE_CLASSES     = {"A", "B", "C", "D", "E", "F", "G"}
VALID_IGNITION_CAUSES  = {"Lightning", "Human", "Unknown"}
VALID_DURATION_CLASSES = {"< 1 day", "1-7 days", "8-30 days", "> 30 days", None}

In [ ]:
NUMERICAL_BOUNDS = {
    "final_size_acres":        (0.0,    700_000.0),
    "containment_days":        (0.0,    730.0),      # 2 years max (extreme outlier threshold)
    "size_log":                (0.0,    6.0),
    "weather.max_temp_c":      (-30.0,  60.0),
    "weather.wind_speed_ms":   (0.0,    50.0),
    "weather.relative_humidity": (0.0,  100.0),
    "weather.kbdi":            (0.0,    800.0),
    "weather.erc":             (0.0,    150.0),
}

In [ ]:
def setup_logging(log_path: Path) -> logging.Logger:
    """Configure file + console logging and return the module logger."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s  %(levelname)s  %(message)s",
        handlers=[
            logging.FileHandler(log_path, mode="w"),
            logging.StreamHandler(),
        ],
    )
    return logging.getLogger(__name__)

In [ ]:
def get_nested(doc: dict, dotted_path: str):
    """
    Retrieve a value from a nested document using a dot-notation path.
    Returns None if any intermediate key is missing.

    Example: get_nested(doc, 'weather.max_temp_c') → doc['weather']['max_temp_c']
    """
    parts = dotted_path.split(".")
    val   = doc
    for part in parts:
        if not isinstance(val, dict):
            return None
        val = val.get(part)
    return val


def validate_required_fields(doc: dict) -> list[str]:
    """
    Check that every required top-level field is present and non-null.

    Returns a list of field names that are missing; empty list = pass.
    """
    missing = []
    for field in REQUIRED_FIELDS:
        if field not in doc or doc[field] is None:
            missing.append(field)
    return missing


def validate_discrete_values(doc: dict) -> list[str]:
    """
    Check that discrete/categorical fields contain only expected values.

    Returns a list of human-readable violation strings; empty list = pass.
    """
    violations = []

    sc = doc.get("size_class")
    if sc not in VALID_SIZE_CLASSES:
        violations.append(f"size_class='{sc}' not in {VALID_SIZE_CLASSES}")

    ic = doc.get("ignition_cause")
    if ic not in VALID_IGNITION_CAUSES:
        violations.append(f"ignition_cause='{ic}' not in {VALID_IGNITION_CAUSES}")

    dc = doc.get("duration_class")
    if dc not in VALID_DURATION_CLASSES:
        violations.append(f"duration_class='{dc}' not in {VALID_DURATION_CLASSES}")

    return violations


def validate_numerical_bounds(doc: dict) -> list[str]:
    """
    Check that numerical fields fall within defined acceptable ranges.
    Fields that are None/missing are not flagged here (covered by required-field check).

    Returns a list of human-readable out-of-range violation strings.
    """
    violations = []
    for dotted_path, (lo, hi) in NUMERICAL_BOUNDS.items():
        val = get_nested(doc, dotted_path)
        if val is None:
            continue   # missing optional field — not a bounds violation
        try:
            fval = float(val)
        except (TypeError, ValueError):
            violations.append(f"{dotted_path}='{val}' is not numeric")
            continue
        if math.isnan(fval):
            violations.append(f"{dotted_path} is NaN")
        elif not (lo <= fval <= hi):
            violations.append(f"{dotted_path}={fval:.4g} outside [{lo}, {hi}]")
    return violations


def validate_location(doc: dict) -> list[str]:
    """
    Check that the GeoJSON location sub-document is well-formed:
    - type == 'Point'
    - coordinates is a 2-element list [lon, lat]
    - values are within valid WGS-84 range

    Returns list of violation strings; empty = pass.
    """
    violations = []
    loc = doc.get("location")
    if loc is None:
        return []   # already caught by required-field check

    if loc.get("type") != "Point":
        violations.append(f"location.type='{loc.get('type')}' expected 'Point'")

    coords = loc.get("coordinates")
    if not isinstance(coords, (list, tuple)) or len(coords) != 2:
        violations.append(f"location.coordinates malformed: {coords}")
        return violations

    lon, lat = coords[0], coords[1]
    if not (-180 <= lon <= 180):
        violations.append(f"location.coordinates longitude={lon} out of [-180, 180]")
    if not (-90 <= lat <= 90):
        violations.append(f"location.coordinates latitude={lat} out of [-90, 90]")

    return violations


def validate_document(doc: dict) -> dict:
    """
    Run all validation checks on a single document and return a result dict.

    Args:
        doc: One MongoDB document from the wildfires collection.

    Returns:
        Dict with keys:
            _id        — document identifier
            passed     — True if no violations found
            violations — list of violation strings
    """
    violations = []
    violations += validate_required_fields(doc)
    violations += validate_discrete_values(doc)
    violations += validate_numerical_bounds(doc)
    violations += validate_location(doc)

    return {
        "_id":        doc.get("_id", "UNKNOWN"),
        "passed":     len(violations) == 0,
        "violations": violations,
    }

In [ ]:
def generate_report(
    results: list[dict],
    df_numeric: pd.DataFrame,
    report_path: Path,
    logger: logging.Logger,
) -> None:
    """
    Write a human-readable validation and statistics report to a text file.

    Args:
        results:     List of validate_document() result dicts.
        df_numeric:  DataFrame of numeric fields from all documents,
                     used to compute descriptive statistics.
        report_path: Destination .txt file.
        logger:      Module logger.
    """
    total    = len(results)
    passed   = sum(1 for r in results if r["passed"])
    failed   = total - passed
    pass_pct = 100 * passed / total if total else 0.0

    lines = [
        "=" * 70,
        "DS 4320 Project 2 — Wildfire Dataset Validation Report",
        f"Generated: {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}",
        f"Collection: {DB_NAME}.{COLLECTION}",
        "=" * 70,
        "",
        "── Document Validation Summary ─────────────────────────────────────",
        f"  Total documents checked : {total:,}",
        f"  Passed                  : {passed:,}  ({pass_pct:.1f}%)",
        f"  Failed                  : {failed:,}",
        "",
    ]

    # Top violation types
    if failed > 0:
        from collections import Counter
        all_violations = [v for r in results for v in r["violations"]]
        top = Counter(all_violations).most_common(10)
        lines.append("── Top Violation Types ──────────────────────────────────────────────")
        for msg, cnt in top:
            lines.append(f"  [{cnt:>6}]  {msg}")
        lines.append("")

    # Descriptive statistics for numeric features
    lines.append("── Numeric Feature Statistics ───────────────────────────────────────")
    if not df_numeric.empty:
        stats = df_numeric.describe().T
        lines.append(stats.to_string())
    else:
        lines.append("  (no numeric data available)")
    lines.append("")

    # Field completeness
    lines.append("── Field Completeness ───────────────────────────────────────────────")
    all_fields = [
        "fire_name", "discovery_date", "containment_date",
        "final_size_acres", "containment_days", "size_log",
        "size_class", "duration_class", "ignition_cause",
        "state", "gacc_region", "weather", "fuel_model", "incident_type",
    ]
    for f in all_fields:
        # Count docs where field is present and non-null from results
        pass   # computed from df_numeric; placeholder for manual addition
    lines.append("  (see logs/invalid_docs.json for full per-document breakdown)")
    lines.append("")
    lines.append("=" * 70)

    report_path.parent.mkdir(parents=True, exist_ok=True)
    with report_path.open("w", encoding="utf-8") as fh:
        fh.write("\n".join(lines))

    logger.info("Validation report written to %s", report_path)

In [ ]:
def run_validation(
    mongo_uri: str,
    db_name: str,
    collection_name: str,
    sample_n: int | None,
    logger: logging.Logger,
) -> None:
    """
    Connect to MongoDB, iterate over the wildfires collection, validate
    every document, and write the report files.

    Args:
        mongo_uri:       MongoDB Atlas connection string.
        db_name:         Target database name.
        collection_name: Target collection name.
        sample_n:        If set, validate only a random sample of this size.
        logger:          Module logger.
    """
    try:
        client = MongoClient(mongo_uri, serverSelectionTimeoutMS=10_000)
        client.admin.command("ping")
        col = client[db_name][collection_name]
        logger.info("Connected to MongoDB Atlas — %s.%s", db_name, collection_name)
    except pymongo.errors.ConnectionFailure as exc:
        logger.error("MongoDB connection failed: %s", exc)
        raise

    total_in_col = col.count_documents({})
    logger.info("Total documents in collection: %d", total_in_col)

    # Build cursor — use $sample for random subset if requested
    if sample_n and sample_n < total_in_col:
        cursor = col.aggregate([{"$sample": {"size": sample_n}}])
        logger.info("Validating random sample of %d documents", sample_n)
    else:
        cursor = col.find({})
        logger.info("Validating all %d documents", total_in_col)

    results      = []
    numeric_rows = []
    invalid_ids  = []

    for doc in cursor:
        result = validate_document(doc)
        results.append(result)

        if not result["passed"]:
            invalid_ids.append({"_id": result["_id"], "violations": result["violations"]})
            logger.warning("FAIL %s — %s", result["_id"], "; ".join(result["violations"]))

        # Collect numeric fields for stats
        row = {
            "_id":               doc.get("_id"),
            "final_size_acres":  doc.get("final_size_acres"),
            "containment_days":  doc.get("containment_days"),
            "size_log":          doc.get("size_log"),
        }
        weather = doc.get("weather") or {}
        row["max_temp_c"]         = weather.get("max_temp_c")
        row["wind_speed_ms"]      = weather.get("wind_speed_ms")
        row["relative_humidity"]  = weather.get("relative_humidity")
        row["kbdi"]               = weather.get("kbdi")
        row["erc"]                = weather.get("erc")
        numeric_rows.append(row)

        # Progress heartbeat
        if len(results) % 50_000 == 0:
            logger.info("  ... validated %d documents", len(results))

    client.close()

    # Build numeric DataFrame for statistics
    df_numeric = pd.DataFrame(numeric_rows).set_index("_id")
    numeric_cols = [c for c in df_numeric.columns if c != "_id"]
    df_numeric[numeric_cols] = df_numeric[numeric_cols].apply(pd.to_numeric, errors="coerce")

    # Write validation report
    generate_report(results, df_numeric[numeric_cols], REPORT_PATH, logger)

    # Write invalid document list
    INVALID_DOCS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with INVALID_DOCS_PATH.open("w", encoding="utf-8") as fh:
        json.dump(invalid_ids, fh, indent=2, default=str)
    logger.info("Wrote %d invalid doc records to %s", len(invalid_ids), INVALID_DOCS_PATH)

    # Final summary to console
    passed = sum(1 for r in results if r["passed"])
    print(f"\n{'='*50}")
    print(f"Validation complete: {passed:,}/{len(results):,} documents passed")
    print(f"Report  → {REPORT_PATH}")
    print(f"Invalid → {INVALID_DOCS_PATH}")
    print(f"Log     → {LOG_PATH}")
    print(f"{'='*50}\n")

In [ ]:
def parse_args() -> argparse.Namespace:
    """Parse command-line arguments."""
    parser = argparse.ArgumentParser(
        description="Validate MongoDB wildfires collection against the project soft schema."
    )
    parser.add_argument(
        "--sample", type=int, default=None,
        help="Validate only a random sample of N documents (default: all)"
    )
    # Use parse_known_args to ignore arguments passed by the Colab kernel
    args, unknown = parser.parse_known_args()
    return args


def main() -> None:
    """Main entry point — orchestrates validation and reporting."""
    args   = parse_args()
    logger = setup_logging(LOG_PATH)
    logger.info("=== 04_validate_schema.py started ===")

    try:
        run_validation(MONGO_URI, DB_NAME, COLLECTION, args.sample, logger)
    except Exception as exc:
        logger.error("Fatal error: %s", exc)
        raise

    logger.info("=== 04_validate_schema.py finished ===")


if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.


In [ ]:
import requests

try:
    response = requests.get('http://httpbin.org/ip')
    ip_data = response.json()
    print(f"Your public IP address is: {ip_data['origin']}")
except requests.exceptions.RequestException as e:
    print(f"Could not retrieve IP address: {e}")

Your public IP address is: 136.114.153.104
